# L06 – Utvärdering: mått, inlärningskurvor och felanalys

**Tid:** ca 2 h · **Del B: Träna ett nätverk väl**

## Mål
1. Välja **mått** efter problemtyp och veta när accuracy är missvisande.
2. Läsa av **inlärningskurvor** och känna igen **över- och underanpassning**.
3. Göra en enkel **felanalys**: confusion matrix och de värsta felen.
4. Utvärdera på testmängden **en gång**, i slutet.

**Var i arbetsflödet?** Steg 1 (mått), 7 (diagnos) och 9 (test). Det mesta känner du igen från `sklearn.metrics`. Det nya är att vi följer kurvorna *under* träningen.

> Kräver `dlkit` från L05.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from dlkit.seed import set_seed
from dlkit.data import fashion_mnist_loaders, FASHION_CLASSES
from dlkit.train import fit, evaluate, predict
from dlkit.plots import plot_history

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
loss_fn = nn.CrossEntropyLoss()

def make_mlp(hidden_sizes):
    layers, in_features = [nn.Flatten()], 784
    for h in hidden_sizes:
        layers += [nn.Linear(in_features, h), nn.ReLU()]
        in_features = h
    layers.append(nn.Linear(in_features, 10))
    return nn.Sequential(*layers)

Vi tränar MLP:n från L05 i 10 epoker. Den är modellen vi utvärderar i resten av lektionen.

In [ ]:
train_loader, val_loader, test_loader = fashion_mnist_loaders()
set_seed(42)
model = make_mlp([256, 128]).to(device)
history = fit(model, train_loader, val_loader, loss_fn,
              torch.optim.Adam(model.parameters(), lr=1e-3), device, epochs=10)

## 1. Mått för regression

| Mått | Formel | Egenskap | När |
|---|---|---|---|
| **MAE** | $\frac{1}{N}\sum \lvert y - \hat y \rvert$ | i målets enhet, robust mot outliers | när alla fel ska räknas lika |
| **RMSE** | $\sqrt{\frac{1}{N}\sum (y - \hat y)^2}$ | i målets enhet, straffar stora fel hårt | när stora fel är särskilt dåliga |
| **R²** | $1 - \frac{\sum (y-\hat y)^2}{\sum (y - \bar y)^2}$ | andel förklarad varians: 1 är perfekt, 0 är som att gissa medel | för att jämföra mellan problem |

Ett litet exempel visar skillnaden mellan MAE och RMSE när det finns **ett stort fel**:

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

y_true = np.array([10, 20, 15, 30, 25, 12])
pred_a = np.array([11, 19, 16, 29, 26, 11])    # alla fel är 1
pred_b = np.array([10, 20, 15, 30, 25, 18])    # ett fel på 6, resten perfekta

for name, p in [("A: många små fel", pred_a), ("B: ett stort fel", pred_b)]:
    print(f"{name:18s} MAE={mean_absolute_error(y_true, p):.2f}  "
          f"RMSE={root_mean_squared_error(y_true, p):.2f}  R²={r2_score(y_true, p):.2f}")

Båda har samma MAE (1,0), men RMSE är 2,4 gånger högre för B, och R² sjunker också. **Välj mått efter vad ett fel kostar i verkligheten.** I PyTorch tränar man ofta med `MSELoss` men *rapporterar* MAE eller RMSE, eftersom de är lättare att tolka.

## 2. Mått för klassificering

`dlkit.predict` ger `y_true`, `y_pred` och sannolikheterna som NumPy-arrayer. Då kan vi använda `sklearn.metrics` precis som vanligt:

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_true, y_pred, probs = predict(model, val_loader, device)
print(f"val accuracy: {accuracy_score(y_true, y_pred):.3f}\n")
print(classification_report(y_true, y_pred, target_names=FASHION_CLASSES, digits=3))

**Påminnelse:**
- **Precision**: av allt modellen kallade "Shirt", hur stor andel var en skjorta?
- **Recall**: av alla riktiga skjortor, hur stor andel hittade modellen?
- **F1** är det harmoniska medelvärdet av de två.

Lägg märke till att **Shirt** har klart lägst precision och recall av alla klasser. Accuracy (ett enda tal) döljer det.

- **Macro avg**: medel över klasserna, där alla klasser väger lika.
- **Weighted avg**: viktat med antalet exempel per klass.

### När är accuracy missvisande?
När klasserna är **obalanserade**. Vi gör om problemet till "är bilden en väska (Bag) eller inte?". Bara 10 % är väskor:

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

is_bag_true = (y_true == 8).astype(int)
always_no = np.zeros_like(is_bag_true)            # en "modell" som aldrig säger väska
is_bag_pred = (y_pred == 8).astype(int)           # vår MLP

for name, p in [("Säger alltid 'nej'", always_no), ("Vår MLP", is_bag_pred)]:
    print(f"{name:20s} accuracy={accuracy_score(is_bag_true, p):.3f}  "
          f"recall={recall_score(is_bag_true, p, zero_division=0):.3f}  "
          f"F1={f1_score(is_bag_true, p, zero_division=0):.3f}")

Den värdelösa modellen får **90 % accuracy**, men recall och F1 är 0. **Tumregel:** vid obalanserade klasser rapporterar du precision, recall och F1 (eller PR-AUC), inte bara accuracy.

| Problem | Börja med | Lägg till när … |
|---|---|---|
| Balanserad klassificering | accuracy | per-klass-rapport för att hitta svaga klasser |
| Obalanserad klassificering | F1, precision och recall | kostnaden för falska positiva och falska negativa skiljer sig åt |
| Regression | MAE eller RMSE | R² för jämförelse |

## 3. Inlärningskurvor: över- och underanpassning

I djupinlärning är **inlärningskurvan**, alltså loss per epok för train och val, det viktigaste diagnosverktyget. Den motsvarar sklearns `validation_curve`, men längs träningstiden:

In [ ]:
plot_history(history, "MLP [256, 128] på 50 000 bilder")

| Mönster | Diagnos | Vad du gör |
|---|---|---|
| Train **och** val loss höga, nära varandra | **Underanpassning**: modellen är för enkel eller har tränat för kort | större modell, fler epoker, högre learning rate |
| Train loss sjunker, val loss vänder **uppåt** | **Överanpassning**: modellen lär sig träningsdatan utantill | regularisering (L07), mer data, augmentation (L10), early stopping |
| Båda sjunker fortfarande | inte klar | träna längre |

Vi framkallar båda fallen avsiktligt:

In [ ]:
# Underanpassning: en pytteliten modell (4 dolda neuroner)
set_seed(42)
tiny = make_mlp([4]).to(device)
h_under = fit(tiny, train_loader, val_loader, loss_fn,
              torch.optim.Adam(tiny.parameters(), lr=1e-3), device, epochs=10, verbose=False)
plot_history(h_under, "Underanpassning: för liten modell")

In [ ]:
# Överanpassning: stor modell, bara 2 000 träningsbilder, många epoker
small_train, small_val, _ = fashion_mnist_loaders(train_subset=2000)
set_seed(42)
big = make_mlp([512, 512]).to(device)
h_over = fit(big, small_train, small_val, loss_fn,
             torch.optim.Adam(big.parameters(), lr=1e-3), device, epochs=40, verbose=False)
plot_history(h_over, "Överanpassning: stor modell, lite data")
print(f"bästa val loss vid epok {int(np.argmin(h_over['val_loss'])) + 1}")

Lägg märke till två saker i överanpassningsfallet:
1. **Train accuracy når nästan 100 %**, medan val accuracy planar ut runt 83 %. Det stora gapet är det tydligaste tecknet.
2. **Val loss vänder uppåt långt innan val accuracy sjunker.** Modellen blir alltmer *självsäker* på sina fel. Därför tittar vi främst på val **loss** när vi letar efter överanpassning.

## 4. Felanalys

Ett mått säger *hur mycket* fel modellen gör, men inte *vilka* fel. Felanalys ger dig idéer om vad du ska förbättra.

### Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=FASHION_CLASSES).plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("Confusion matrix (val)"); plt.show()

Rader är den rätta klassen och kolumner prediktionen. Utanför diagonalen syns tydliga mönster: **Shirt** förväxlas med T-shirt, Pullover och Coat. Det är rimligt, eftersom de ser lika ut i 28×28 pixlar. Sandal, Sneaker och Ankle boot förväxlas också med varandra.

### De mest självsäkra felen
Fel som modellen är *säker* på är extra intressanta. De kan vara riktigt svåra fall, eller **felmärkt data**.

In [ ]:
val_images = torch.stack([val_loader.dataset[i][0] for i in range(len(val_loader.dataset))])
wrong = np.where(y_pred != y_true)[0]
confidence = probs[wrong, y_pred[wrong]]
worst = wrong[np.argsort(-confidence)[:12]]

fig, axes = plt.subplots(2, 6, figsize=(13, 5))
for ax, i in zip(axes.flat, worst):
    ax.imshow(val_images[i].squeeze(), cmap="gray")
    ax.set_title(f"sann: {FASHION_CLASSES[y_true[i]]}\ngissad: {FASHION_CLASSES[y_pred[i]]}\n({probs[i, y_pred[i]]:.0%})", fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()

Fundera på vad du ser. Är det fel du själv också skulle göra? Ser någon etikett felaktig ut? Om felen gäller former och detaljer är det ett tecken på att en modell som förstår bildstruktur (CNN, L10) kan hjälpa.

## 5. Testmängden – en gång

Vi har valt modell med hjälp av val. Nu gör vi steg 9 i arbetsflödet. Det här görs **en gång**, och resultatet rapporteras. Vi går inte tillbaka och justerar efteråt.

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)
print(f"TEST accuracy: {test_acc:.3f}  (val var {history['val_acc'][-1]:.3f})")

Testresultatet hamnar nära val. Det betyder att valideringsmängden gav en ärlig bild. Om test hade blivit klart sämre hade det varit ett tecken på att vi överanpassat våra val till valideringsmängden, eller att testdatan skiljer sig från träningsdatan.

## 6. Övningar

### Övning 1 – precision och recall från confusion matrix
Beräkna precision och recall för klassen **Shirt** (index 6) direkt från `cm` och kontrollera mot sklearn.

*Ledtråd:* precision använder en kolumn och recall en rad.

In [ ]:
shirt_precision = None  # TODO
shirt_recall = None     # TODO

In [ ]:
assert np.isclose(shirt_precision, precision_score(y_true, y_pred, labels=[6], average="macro"))
assert np.isclose(shirt_recall, recall_score(y_true, y_pred, labels=[6], average="macro"))
print(f"Rätt! precision={shirt_precision:.3f} recall={shirt_recall:.3f}")

### Övning 2 – de vanligaste förväxlingarna
Hitta de **3 vanligaste felen** som par (sann klass, gissad klass) med antal, till exempel `("Shirt", "T-shirt/top", 150)`.

*Ledtråd:* nollställ diagonalen i en kopia av `cm`.

In [ ]:
top3 = None  # TODO: lista med 3 tupler (sann, gissad, antal), sorterad med störst först

In [ ]:
assert len(top3) == 3 and top3[0][2] >= top3[1][2] >= top3[2][2]
assert "Shirt" in (top3[0][0], top3[0][1])
print("Rätt!", top3)

### Övning 3 – diagnos
Vilken diagnos (under- eller överanpassning, eller inte klar) passar, och vad gör du?
1. Train loss 0,05, val loss 0,90, val loss steg från epok 8.
2. Train accuracy 72 %, val accuracy 71 %, båda har legat stilla de senaste 10 epokerna.
3. Train och val loss sjunker fortfarande tydligt efter 5 epoker.

*Dina svar:*

1.
2.
3.

## När använder jag detta?
- **Inlärningskurvor:** i *varje* träning. Plotta både loss och mått för train och val.
- **Val loss** för att upptäcka överanpassning tidigt (före accuracy).
- **Per-klass-rapport och confusion matrix:** när du vill veta *var* modellen brister, och alltid vid obalanserade klasser.
- **De mest självsäkra felen:** när du vill hitta svåra fall eller felmärkt data.
- **Testmängden:** en gång, för den slutgiltiga modellen.

## Sammanfattning
- Välj mått efter problem och kostnad. Accuracy är missvisande vid obalans.
- Ett stort gap mellan train och val betyder överanpassning, och att båda är dåliga betyder underanpassning.
- Felanalys pekar ut vad du ska förbättra härnäst.

**Nästa lektion:** regularisering, alltså verktygen mot överanpassningen vi just framkallade.